# This Notebook will ingest Fusion data extracted by BICC to Silver Catalog

In [41]:
# Define parameters
target_type   =oidlUtils.parameters.getParameter("TARGET_TYPE", "table")
target_format =oidlUtils.parameters.getParameter("TARGET_FORMAT", "parquet")
bronze_folder_path    = "/Volumes/demo_fusion_bronze/demo_fusion_bronze_schema/demo_fusion_bronze_vol"
silver_catalog    = "demo_fusion_silver"
silver_schema     = "demo_fusion_silver_schema"

 Trigger Fusion BICC from AIDP (use this code if you want to trigger BICC from AI Data Platform.)
spark.read.format("aidataplatform") \
    .option("type", "FUSION_BICC") \
    .option("fusion.service.url", "https://fa-etat..........com/") \
    .option("user.name", "<user name") \
    .option("password", <password>) \
    .option("schema", "Financial") \
    .option("fusion.external.storage", "<bicc extrenal storage> ") \
    .option("datastore", "FscmTopModelAM.PrcExtractAM.PozBiccExtractAM.SupplierExtractPVO") \
    .load().show()

# Ingest AP Invoice Header data into Silver Catalog

In [42]:
df_raw = spark.read \
    .option("header", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("delimiter", ",") \
    .option("multiLine", "false") \
    .option("encoding", "UTF-8") \
	.csv(f"{bronze_folder_path}/file_fscmtopmodelam_finextractam_apbiccextractam_invoiceheaderextractpvo-batch1320709083-20251103_170429.csv.gz")

In [43]:
fusion_invoice_header_df = df_raw.select(
    "APINVOICESINVOICEID",
    "APINVOICESINVOICENUM",
    "APINVOICESVENDORID",
    "APINVOICESORGID",
    "APINVOICESINVOICEDATE",
    "APINVOICESINVOICERECEIVEDDATE",
    "APINVOICESINVOICEAMOUNT",
    "APINVOICESBASEAMOUNT",
    "APINVOICESINVOICECURRENCYCODE",
    "APINVOICESTERMSID",
    "APINVOICESPAYMENTSTATUSFLAG",
    "APINVOICESAPPROVALSTATUS",
    "APINVOICESCREATIONDATE",
    "APINVOICESLASTUPDATEDATE"
)

In [44]:
from pyspark.sql.functions import col, to_date, StringType
from pyspark.sql.types import DoubleType, DateType

fusion_invoice_header_df = fusion_invoice_header_df.select(
    col("APINVOICESINVOICEID").cast(StringType()),
    col("APINVOICESINVOICENUM").cast(StringType()),
    col("APINVOICESVENDORID").cast(StringType()),
    col("APINVOICESORGID").cast(StringType()),
    to_date(col("APINVOICESINVOICEDATE")).alias("APINVOICESINVOICEDATE"),
    to_date(col("APINVOICESINVOICERECEIVEDDATE")).alias("APINVOICESINVOICERECEIVEDDATE"),
    col("APINVOICESINVOICEAMOUNT").cast(DoubleType()),
    col("APINVOICESBASEAMOUNT").cast(DoubleType()),
    col("APINVOICESINVOICECURRENCYCODE").cast(StringType()),
    col("APINVOICESTERMSID").cast(StringType()),
    col("APINVOICESPAYMENTSTATUSFLAG").cast(StringType()),
    col("APINVOICESAPPROVALSTATUS").cast(StringType()),
    to_date(col("APINVOICESCREATIONDATE")).alias("APINVOICESCREATIONDATE"),
    to_date(col("APINVOICESLASTUPDATEDATE")).alias("APINVOICESLASTUPDATEDATE")
)

In [45]:
fusion_invoice_header_df.write.mode("overwrite").format("parquet").saveAsTable(f"{silver_catalog}.{silver_schema}.fusion_invoice_header")

# Ingest AP Invoice Lines data into Silver Catalog

In [46]:
df_raw1 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(f"{bronze_folder_path}/file_fscmtopmodelam_finextractam_apbiccextractam_invoicelineextractpvo-batch1320709083-20251103_170417.csv.gz")

df_raw2 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(f"{bronze_folder_path}/file_fscmtopmodelam_finextractam_apbiccextractam_invoicelineextractpvo-batch1320709083-20251103_170732.csv.gz")

df_raw=df_raw1.unionByName(df_raw2)

In [47]:
fusion_invoice_line_df = df_raw.select(
    col("APINVOICELINESALLINVOICEID").alias("APINVOICELINESINVOICEID"),
    col("APINVOICELINESALLPOLINEID").alias("APINVOICELINESINVOICELINEID"),
    col("APINVOICELINESALLAMOUNT").alias("APINVOICELINESLINEAMOUNT"),
    col("APINVOICELINESALLPAQUANTITY").alias("APINVOICELINESQUANTITY"),   # Corrected column name
    col("APINVOICELINESALLUNITPRICE").alias("APINVOICELINESUNITPRICE"),
    col("APINVOICELINESALLDESCRIPTION").alias("APINVOICELINESITEMDESCRIPTION"),
    to_date(col("APINVOICELINESALLEXPENDITUREITEMDATE")).alias("APINVOICELINESEXPENDITUREITEMDATE")
)

In [48]:
fusion_invoice_line_df = fusion_invoice_line_df.select(
    col("APINVOICELINESINVOICEID").cast(StringType()),
    col("APINVOICELINESINVOICELINEID").cast(StringType()),
    col("APINVOICELINESLINEAMOUNT").cast(DoubleType()),
    col("APINVOICELINESQUANTITY").cast(DoubleType()),
    col("APINVOICELINESUNITPRICE").cast(DoubleType()),
    col("APINVOICELINESITEMDESCRIPTION").cast(StringType()),
    col("APINVOICELINESEXPENDITUREITEMDATE").cast(DateType())
)

In [49]:
fusion_invoice_line_df.write.mode("overwrite").format("parquet").saveAsTable(f"{silver_catalog}.{silver_schema}.fusion_invoice_line")

# Ingest AP Invoice Payment History to Silver Catalog

In [72]:
# ============================================================
# Ingest AP Payment History Distribution data into Silver
# ============================================================

from pyspark.sql.functions import col, to_date
from pyspark.sql.types import StringType, DoubleType, DateType

# ------------------------------------------------------------
# Read Fusion AP Payment History Distribution extract
# IMPORTANT: delimiter is COMMA, not pipe
# ------------------------------------------------------------

fusion_payment_history_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(
        f"{bronze_folder_path}/"
        "file_fscmtopmodelam_finextractam_apbiccextractam_"
        "paymenthistorydistributionextractpvo-"
        "batch1320709083-20251103_170428.csv.gz"
    )

# ------------------------------------------------------------
# (Optional but recommended) Validate schema
# ------------------------------------------------------------

fusion_payment_history_df.printSchema()

# ------------------------------------------------------------
# Select, rename, and cast columns for Silver
# ------------------------------------------------------------

fusion_payment_history_df = fusion_payment_history_df.select(
    col("APPAYMENTHISTDISTSINVOICEPAYMENTID")
        .cast(StringType())
        .alias("APPAYMENTHISTORYPAYMENTID"),

    col("APPAYMENTHISTDISTSINVOICEDISTRIBUTIONID")
        .cast(StringType())
        .alias("APPAYMENTHISTORYINVOICEID"),

    to_date(col("APPAYMENTHISTDISTSLASTUPDATEDATE"))
        .alias("APPAYMENTHISTORYPAYMENTDATE"),

    col("APPAYMENTHISTDISTSAMOUNT")
        .cast(DoubleType())
        .alias("APPAYMENTHISTORYAMOUNT"),

    col("APPAYMENTHISTDISTSPAYDISTLOOKUPCODE")
        .cast(StringType())
        .alias("APPAYMENTHISTORYSTATUS")
)

# ------------------------------------------------------------
# Write to Silver Catalog
# ------------------------------------------------------------

fusion_payment_history_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .saveAsTable(
        f"{silver_catalog}.{silver_schema}.fusion_payment_history"
    )


root
 |-- APPAYMENTHISTDISTSACCOUNTINGEVENTID: integer (nullable = true)
 |-- APPAYMENTHISTDISTSAMOUNT: double (nullable = true)
 |-- APPAYMENTHISTDISTSAMOUNTVARIANCE: string (nullable = true)
 |-- APPAYMENTHISTDISTSAWTRELATEDID: integer (nullable = true)
 |-- APPAYMENTHISTDISTSBANKCURRAMOUNT: double (nullable = true)
 |-- APPAYMENTHISTDISTSCLEAREDBASEAMOUNT: double (nullable = true)
 |-- APPAYMENTHISTDISTSCREATEDBY: string (nullable = true)
 |-- APPAYMENTHISTDISTSCREATIONDATE: timestamp (nullable = true)
 |-- APPAYMENTHISTDISTSHISTORICALFLAG: string (nullable = true)
 |-- APPAYMENTHISTDISTSINVOICEADJUSTMENTEVENTID: string (nullable = true)
 |-- APPAYMENTHISTDISTSINVOICEBASEAMTVARIANCE: string (nullable = true)
 |-- APPAYMENTHISTDISTSINVOICEBASEQTYVARIANCE: double (nullable = true)
 |-- APPAYMENTHISTDISTSINVOICEDISTAMOUNT: double (nullable = true)
 |-- APPAYMENTHISTDISTSINVOICEDISTBASEAMOUNT: double (nullable = true)
 |-- APPAYMENTHISTDISTSINVOICEDISTRIBUTIONID: long (nullable = true)


In [73]:
%sql
select * from demo_fusion_silver.demo_fusion_silver_schema.fusion_payment_history

# Ingest AP Invoice Payment Term Header to Silver Catalog

In [52]:
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(f"{bronze_folder_path}/file_fscmtopmodelam_finextractam_apbiccextractam_paymenttermheaderextractpvo-batch1320709083-20251103_170328.csv.gz")

df_raw.printSchema()
df_raw.select("PAYMENTTERMHEADERBPEORANK").show(100)

In [53]:
fusion_payment_term_df = df_raw.select(
    col("PAYMENTTERMHEADERBPEOTERMID").alias("APPAYMENTTERMSID"),               # Payment Term ID
    col("PAYMENTTERMHEADERBPEOTYPE").alias("APPAYMENTTERMSTYPE"),               # Term Type
    col("PAYMENTTERMHEADERBPEORANK").alias("APPAYMENTTERMSRANK"),               # Rank or Priority
    to_date(col("PAYMENTTERMHEADERBPEOCREATIONDATE")).alias("APPAYMENTTERMSCREATIONDATE"),  # Creation Date
    to_date(col("PAYMENTTERMHEADERBPEOLASTUPDATEDATE")).alias("APPAYMENTTERMSLASTUPDATEDATE"), # Last Update Date
    col("PAYMENTTERMHEADERBPEOCREATEDBY").alias("APPAYMENTTERMSCREATEDBY")      # Created By
)


In [54]:
fusion_payment_term_df.write.mode("overwrite").format("parquet").saveAsTable(f"{silver_catalog}.{silver_schema}.fusion_payment_term_header")

In [55]:
%sql
select * from demo_fusion_silver.demo_fusion_silver_schema.fusion_payment_term_header

df.show(10, truncate=False)
df.printSchema()